# Tarea 2 - Pregunta de Negocio 1

## Limpieza y transformación de datos

### Objetivo

En este notebook se realiza el proceso de limpieza y transformación de los datos de contratación de INVIAS, tomando como punto de partida los hallazgos obtenidos en la auditoría inicial.

El proceso busca preparar la información necesaria para estudiar la concentración de la contratación por proveedor, modalidad de contratación y tipo de contrato.

Las decisiones de limpieza estarán orientadas a:

- Estandarizar tipos de datos.
- Homologar valores no informativos.
- Estandarizar la identificación de proveedores.
- Conservar la trazabilidad con los registros originales.
- Identificar contratos formalizados.
- Diferenciar registros aptos para análisis por número de contratos y por valor contratado.
- Generar una base limpia para la etapa posterior de alistamiento.

Los indicadores de concentración y las variables analíticas se construirán en el siguiente notebook.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Cargamos la base original de contratos de INVIAS

df = pd.read_csv(
    "../../invias.csv",
    low_memory=False
)

print("Base cargada correctamente.")

Base cargada correctamente.


### 2.1 Creación del conjunto de trabajo

Para conservar la integridad de la base original se crea una copia del conjunto de datos.

Todas las transformaciones realizadas durante la limpieza serán aplicadas sobre esta copia, permitiendo mantener los datos originales disponibles para validaciones posteriores.

In [3]:
# Variables necesarias para la limpieza y análisis de P1

variables_limpieza_p1 = [
    "id_contrato",
    "referencia_del_contrato",
    "proveedor_adjudicado",
    "documento_proveedor",
    "tipodocproveedor",
    "valor_del_contrato",
    "modalidad_de_contratacion",
    "tipo_de_contrato",
    "estado_contrato",
    "fecha_de_firma",
    "es_grupo",
    "es_pyme"
]

# Creamos una copia independiente

df_limpio = df[variables_limpieza_p1].copy()

print("Registros:", len(df_limpio))
print("Variables:", df_limpio.shape[1])

df_limpio.head()

Registros: 25605
Variables: 12


,id_contrato,referencia_del_contrato,proveedor_adjudicado,documento_proveedor,tipodocproveedor,valor_del_contrato,modalidad_de_contratacion,tipo_de_contrato,estado_contrato,fecha_de_firma,es_grupo,es_pyme
0,CO1.PCCNTR.770322,000168-2019,ANGIE GERALDINE RINCÓN CALA,1016046149,Cédula de Ciudadanía,23500000.0,Contratación directa,Prestación de servicios,Cerrado,2019-01-29T00:00:00.000,No,No
1,CO1.PCCNTR.5481469,4028 DE 2023,Diana Catalina Burbano Delgado,1061699354,Cédula de Ciudadanía,15770000.0,Contratación directa,Prestación de servicios,Cerrado,2023-10-23T00:00:00.000,No,No
2,CO1.PCCNTR.6706615,3429 DE 2024,SERGIO ALEXANDER RAMIREZ YAÑES,79051510,Cédula de Ciudadanía,31500000.0,Contratación directa,Prestación de servicios,Cerrado,2024-08-30T00:00:00.000,No,No
3,CO1.PCCNTR.4412417,265 DE 2023,JUAN PABLO CASTILLO VARGAS,1010168538,Cédula de Ciudadanía,17700000.0,Contratación directa,Prestación de servicios,En ejecución,2023-01-16T00:00:00.000,No,No
4,CO1.PCCNTR.3516681,CO1.PCCNTR.3516681,Sin Descripcion,No Definido,No Definido,0.0,Contratación directa,Prestación de servicios,Cancelado,NaN,No,No


### 2.2 Conversión de la fecha de firma

La auditoría inicial mostró que `fecha_de_firma` se encuentra almacenada como texto.

Debido a que esta variable será utilizada para identificar contratos formalizados y posteriormente realizar análisis temporales, se transforma al formato `datetime`.

Los valores que no puedan interpretarse como fechas válidas se representan como valores faltantes.

In [4]:
# Conversión de la fecha de firma

df_limpio["fecha_de_firma"] = pd.to_datetime(df_limpio["fecha_de_firma"],errors="coerce")

print("Fechas válidas:",df_limpio["fecha_de_firma"].notna().sum())

print("Fechas faltantes:",df_limpio["fecha_de_firma"].isna().sum())


print("Fecha mínima:",df_limpio["fecha_de_firma"].min())

print("Fecha máxima:",df_limpio["fecha_de_firma"].max())

Fechas válidas: 17477
Fechas faltantes: 8128
Fecha mínima: 2017-11-29 00:00:00
Fecha máxima: 2026-08-06 00:00:00


### 2.3 Tratamiento de valores no informativos en proveedor adjudicado

Durante la auditoría se identificaron valores textuales que no representan un proveedor identificable, particularmente `Sin Descripcion` y `No Definido`.
Estos valores se homologan como información faltante (`pd.NA`) con el propósito de evitar que sean interpretados posteriormente como proveedores reales durante las agrupaciones y cálculos de concentración.
La transformación se realiza únicamente sobre la variable utilizada para el análisis y no modifica la fuente original.

In [5]:
# Estandarizamos inicialmente el texto

df_limpio["proveedor_adjudicado"] = (df_limpio["proveedor_adjudicado"].astype("string").str.strip())

# Valores que no representan un proveedor identificable

valores_no_informativos_proveedor = [
    "Sin Descripcion",
    "Sin Descripción",
    "No definido",
    "No Definido",
    ""]

df_limpio["proveedor_adjudicado"] = (
    df_limpio["proveedor_adjudicado"]
    .replace(
        valores_no_informativos_proveedor,
        pd.NA
    )
)

print("Valores faltantes después de homologación:",df_limpio["proveedor_adjudicado"].isna().sum())

Valores faltantes después de homologación: 5680


### 2.4 Normalización del nombre del proveedor

La auditoría mostró pequeñas diferencias de formato entre los nombres de los proveedores.

Para disminuir la posibilidad de considerar como proveedores diferentes registros que únicamente varían por el uso de mayúsculas, minúsculas o espacios, se crea una nueva variable denominada `proveedor_normalizado`.

Se conserva `proveedor_adjudicado` como variable original de referencia.

In [6]:
# Creamos una versión estandarizada del nombre

df_limpio["proveedor_normalizado"] = (
    df_limpio["proveedor_adjudicado"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.upper()
)

print("Proveedores originales:",df_limpio["proveedor_adjudicado"].nunique(dropna=True))

print("Proveedores después de normalización:",df_limpio["proveedor_normalizado"].nunique(dropna=True))

Proveedores originales: 9157
Proveedores después de normalización: 9144


### 2.5 Estandarización del documento del proveedor

Debido a que el análisis requiere agrupar correctamente los contratos por proveedor, se utiliza `documento_proveedor` como variable auxiliar de identificación.

Se crea una versión estandarizada del documento eliminando espacios, puntos y guiones. Los valores evidentemente no informativos se transforman en valores faltantes.

Cuando exista un documento válido, este tendrá prioridad para identificar al proveedor. Cuando no exista, se utilizará el nombre normalizado.

In [7]:
# Convertimos el documento a texto y estandarizamos formato

df_limpio["documento_proveedor_limpio"] = (
    df_limpio["documento_proveedor"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Valores textuales que no identifican realmente al proveedor

documentos_no_informativos = [
    "NO DEFINIDO",
    "NO APLICA",
    "DOCUMENTO CONSORCIAL",
    "",
    "NAN"
]

df_limpio["documento_proveedor_limpio"] = (
    df_limpio["documento_proveedor_limpio"]
    .replace(
        documentos_no_informativos,
        pd.NA
    )
)

In [8]:
# Eliminamos puntos, guiones y espacios del documento estandarizado

df_limpio["documento_proveedor_limpio"] = (
    df_limpio["documento_proveedor_limpio"]
    .str.replace(
        r"[\.\-\s]",
        "",
        regex=True
    )
)

# Identificamos documentos compuestos únicamente por ceros

documento_solo_ceros = (
    df_limpio["documento_proveedor_limpio"]
    .fillna("")
    .str.fullmatch(r"0+")
)

df_limpio.loc[
    documento_solo_ceros,
    "documento_proveedor_limpio"
] = pd.NA

### 2.6 Construcción de un identificador analítico del proveedor

Para reducir errores derivados de variaciones en los nombres de los proveedores se construye `proveedor_id`.

La identificación se realiza siguiendo el siguiente criterio:

1. Cuando existe un documento de proveedor utilizable, se emplea el documento como identificador principal.
2. Cuando el documento no está disponible, se utiliza el nombre normalizado del proveedor.
3. Si no existe ninguna de las dos fuentes de identificación, el registro permanece sin identificador de proveedor.

Este procedimiento permite mejorar la consistencia de las futuras agrupaciones sin modificar los campos originales.

In [9]:
# Inicializamos el identificador del proveedor

df_limpio["proveedor_id"] = pd.NA

# Caso 1: existe documento del proveedor

mask_documento = (df_limpio["documento_proveedor_limpio"].notna())

df_limpio.loc[mask_documento,"proveedor_id"] = ("DOC_"+df_limpio.loc[mask_documento,"documento_proveedor_limpio"])

In [10]:
# Caso 2: no hay documento, pero existe nombre del proveedor

mask_nombre = (df_limpio["proveedor_id"].isna()&df_limpio["proveedor_normalizado"].notna())

df_limpio.loc[mask_nombre,"proveedor_id"] = ("NOMBRE_"+df_limpio.loc[mask_nombre,"proveedor_normalizado"])

print("Registros sin proveedor identificable:",df_limpio["proveedor_id"].isna().sum())

Registros sin proveedor identificable: 5678


In [11]:
# El nombre normalizado será la etiqueta principal

df_limpio["proveedor_etiqueta"] = (df_limpio["proveedor_normalizado"])

# Si no existe nombre pero sí documento, utilizamos el documento como etiqueta

mask_sin_nombre_con_documento = (df_limpio["proveedor_etiqueta"].isna()&df_limpio["documento_proveedor_limpio"].notna())

df_limpio.loc[mask_sin_nombre_con_documento,"proveedor_etiqueta"] = ("PROVEEDOR ID "+df_limpio.loc[mask_sin_nombre_con_documento,"documento_proveedor_limpio"])

### 2.8 Estandarización de variables categóricas

Se eliminan espacios adicionales en las variables categóricas relevantes para evitar diferencias generadas únicamente por formato.

Las categorías oficiales de modalidad y tipo de contrato se conservan, debido a que la auditoría no evidenció duplicaciones derivadas únicamente de mayúsculas o espacios.

En el estado contractual se homologa únicamente la capitalización de algunas categorías.

In [12]:
# Variables categóricas a estandarizar

variables_categoricas = [
    "modalidad_de_contratacion",
    "tipo_de_contrato",
    "estado_contrato",
    "es_grupo",
    "es_pyme",
    "tipodocproveedor"
]

for columna in variables_categoricas:
    df_limpio[columna] = (
        df_limpio[columna]
        .astype("string")
        .str.strip()
    )

In [13]:
# Homologamos únicamente diferencias de capitalización en estado

homologacion_estado = {
    "terminado": "Terminado",
    "cedido": "Cedido",
    "enviado Proveedor": "Enviado Proveedor"
}

df_limpio["estado_contrato"] = (
    df_limpio["estado_contrato"]
    .replace(homologacion_estado)
)

df_limpio["estado_contrato"].value_counts(
    dropna=False
)

estado_contrato
Cerrado              9263
Modificado           5434
Terminado            3416
En ejecución         2512
Borrador             2438
Aprobado              872
Cancelado             629
Suspendido            361
Enviado Proveedor     301
En aprobación         287
Cedido                 92
Name: count, dtype: Int64

### 2.9 Tratamiento de registros duplicados

La auditoría inicial mostró que los 25.605 registros presentan un `id_contrato` único y no existen filas completamente duplicadas.

Por esta razón no se realiza eliminación de registros por duplicidad.

`referencia_del_contrato` no se utilizará como identificador único debido a que presenta valores repetidos y algunas referencias genéricas.

In [14]:
# Validación de duplicados después de las transformaciones

print(
    "ID contrato duplicados:",
    df_limpio["id_contrato"].duplicated().sum()
)

print(
    "Filas completamente duplicadas:",
    df_limpio.duplicated().sum()
)

ID contrato duplicados: 0
Filas completamente duplicadas: 0


### 2.10 Identificación de contratos formalizados

La auditoría mostró que los registros con valor contractual pero sin fecha de firma corresponden exclusivamente a estados preliminares o cancelados.

Asimismo, varios de los valores contractuales extremadamente altos se encuentran dentro de este grupo.

Para el análisis de concentración contractual se considera como contrato formalizado aquel que cuenta con `fecha_de_firma`.

En lugar de eliminar inmediatamente el resto de los registros, se crea inicialmente una variable indicadora que permite documentar claramente este criterio de selección.

In [15]:
# Indicador de contrato formalizado

df_limpio["contrato_formalizado"] = (
    df_limpio["fecha_de_firma"].notna()
)

print(
    "Contratos formalizados:",
    df_limpio["contrato_formalizado"].sum()
)

print(
    "Registros sin firma:",
    (~df_limpio["contrato_formalizado"]).sum()
)

Contratos formalizados: 17477
Registros sin firma: 8128


### 2.11 Tratamiento del valor contractual

La auditoría identificó contratos formalizados cuyo valor contractual es igual a cero.

Debido a que estos registros pueden seguir representando contratos reales, no serán eliminados del conjunto utilizado para los análisis basados en número de contratos.

Sin embargo, un valor igual a cero no aporta información para medir concentración monetaria. Por esta razón se crea una variable indicadora que permitirá utilizar únicamente valores positivos en los análisis basados en monto contratado.

In [16]:
# Identificador de valor contractual positivo

df_limpio["valor_positivo"] = (
    df_limpio["valor_del_contrato"] > 0
)

print(
    "Contratos con valor positivo:",
    df_limpio["valor_positivo"].sum()
)

print(
    "Contratos con valor cero:",
    (df_limpio["valor_del_contrato"] == 0).sum()
)

Contratos con valor positivo: 19632
Contratos con valor cero: 1086


### 2.12 Definición de registros aptos para la Pregunta de Negocio 1

La Pregunta de Negocio 1 estudia la concentración desde dos perspectivas diferentes:

- concentración por número de contratos;
- concentración por valor contratado.

Para evitar perder contratos formalizados con valor igual a cero se definen dos indicadores:

`apto_conteo_p1`: contrato formalizado, con proveedor identificable, modalidad y tipo de contrato disponibles.

`apto_valor_p1`: cumple las condiciones anteriores y adicionalmente presenta un valor contractual positivo.

Esta separación permite utilizar una población coherente para cada tipo de indicador.

In [17]:
# Registros aptos para análisis por número de contratos

df_limpio["apto_conteo_p1"] = (
    df_limpio["contrato_formalizado"]
    &
    df_limpio["proveedor_id"].notna()
    &
    df_limpio["modalidad_de_contratacion"].notna()
    &
    df_limpio["tipo_de_contrato"].notna()
)

# Registros aptos para análisis por valor contractual

df_limpio["apto_valor_p1"] = (
    df_limpio["apto_conteo_p1"]
    &
    df_limpio["valor_positivo"]
)

In [18]:
print(
    "Registros originales:",
    len(df_limpio)
)

print(
    "Aptos para análisis por número de contratos:",
    df_limpio["apto_conteo_p1"].sum()
)

print(
    "Aptos para análisis por valor contratado:",
    df_limpio["apto_valor_p1"].sum()
)

print(
    "Diferencia:",
    (
        df_limpio["apto_conteo_p1"].sum()
        -
        df_limpio["apto_valor_p1"].sum()
    )
)

Registros originales: 25605
Aptos para análisis por número de contratos: 17477
Aptos para análisis por valor contratado: 17244
Diferencia: 233


### 2.13 Construcción del conjunto limpio para P1

Para la siguiente etapa se conserva el universo de contratos formalizados aptos para el análisis por número de contratos.

Los contratos con valor cero permanecen en la base y son identificados mediante `valor_positivo` y `apto_valor_p1`, de manera que puedan excluirse únicamente cuando se calculen indicadores monetarios.

Este criterio permite evitar la pérdida de información útil para los análisis de frecuencia contractual.

In [19]:
# Construimos la base limpia de P1

df_limpio_p1 = (
    df_limpio[
        df_limpio["apto_conteo_p1"]
    ]
    .copy()
)

print(
    "Registros en la base limpia P1:",
    len(df_limpio_p1)
)

Registros en la base limpia P1: 17477


In [20]:
# Validamos variables indispensables

variables_indispensables = [
    "id_contrato",
    "proveedor_id",
    "proveedor_etiqueta",
    "modalidad_de_contratacion",
    "tipo_de_contrato",
    "estado_contrato",
    "fecha_de_firma",
    "valor_del_contrato"
]

df_limpio_p1[
    variables_indispensables
].isna().sum()

id_contrato                  0
proveedor_id                 0
proveedor_etiqueta           0
modalidad_de_contratacion    0
tipo_de_contrato             0
estado_contrato              0
fecha_de_firma               0
valor_del_contrato           0
dtype: int64

In [21]:
# Revisamos nuevamente condiciones importantes

print(
    "Registros finales:",
    len(df_limpio_p1)
)

print(
    "ID contrato únicos:",
    df_limpio_p1["id_contrato"].nunique()
)

print(
    "Proveedores identificados:",
    df_limpio_p1["proveedor_id"].nunique()
)

print(
    "Contratos con valor positivo:",
    df_limpio_p1["valor_positivo"].sum()
)

print(
    "Contratos con valor cero:",
    (df_limpio_p1["valor_del_contrato"] == 0).sum()
)

print(
    "Fecha mínima:",
    df_limpio_p1["fecha_de_firma"].min()
)

print(
    "Fecha máxima:",
    df_limpio_p1["fecha_de_firma"].max()
)

Registros finales: 17477
ID contrato únicos: 17477
Proveedores identificados: 8748
Contratos con valor positivo: 17244
Contratos con valor cero: 233
Fecha mínima: 2017-11-29 00:00:00
Fecha máxima: 2026-08-06 00:00:00


In [22]:
# Resumen del proceso de limpieza

resumen_limpieza = pd.DataFrame({
    "Indicador": [
        "Registros originales",
        "Contratos formalizados",
        "Base limpia para conteo",
        "Registros aptos para valor",
        "Contratos con valor cero conservados"
    ],
    "Resultado": [
        len(df),
        df_limpio["contrato_formalizado"].sum(),
        len(df_limpio_p1),
        df_limpio["apto_valor_p1"].sum(),
        (
            df_limpio_p1["valor_del_contrato"] == 0
        ).sum()
    ]
})

resumen_limpieza

,Indicador,Resultado
0,Registros originales,25605
1,Contratos formalizados,17477
2,Base limpia para conteo,17477
3,Registros aptos para valor,17244
4,Contratos con valor cero conservados,233


### 2.16 Exportación del conjunto limpio

Una vez aplicadas y validadas las reglas de limpieza, se exporta el conjunto resultante.

Esta base constituirá la entrada del notebook de alistamiento, donde se generarán las variables temporales y los indicadores necesarios para medir la concentración contractual.

In [23]:
# Exportamos la base limpia

df_limpio_p1.to_csv(
    "df_limpio_p1.csv",
    index=False
)

print(
    "Base df_limpio_p1.csv exportada correctamente."
)

Base df_limpio_p1.csv exportada correctamente.
